In [ ]:
# IPython magig  tools
%load_ext autoreload
%autoreload 2

from aind_vr_foraging_analysis.utils.parsing import data_access
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from aind_vr_foraging_analysis.utils.plotting  import general_plotting_utils as plotting
import os
results_path =r'C:\Users\tiffany.ona\OneDrive - Allen Institute\Documents\VR foraging\manuscript\results\figures'

In [ ]:
date_string = "2026-03-01" # YYYY-MM-DD
mouse_list = ['823993', '841303', '841314', '815103'] 
            #   ['828418', '828417', '828423', '828420', '828425', '815102', '815103', '815103']
# mouse_list = ['823993']
list_of_epochs = []
agg_vel = []
session_n = 0
for mouse in mouse_list:
    print(f"Processing mouse {mouse}...")
    # This section will look at all the session paths that fulfill the condition
    session_paths = data_access.find_sessions_relative_to_date(
        mouse=mouse,
        date_string=date_string,
        when='on_or_after',
    )
    # Iterate over the session paths and load the data
    for session_path in session_paths:
        # print(f"Loading {session_path.name}...")
        try:
            all_epochs, stream_data, data = data_access.load_session(
                session_path, extra=True
            )
        except Exception as e:
            print(f"Error loading {session_path.name}: {e}")
            continue
        
        print(f"Processing {session_path.name}...")
        all_epochs['mouse'] = mouse
        all_epochs['session_date'] = session_path.name.split('_')[1][:10]
        all_epochs['session_n'] = session_n
        all_epochs['rig'] =  data['config'].streams.rig_input.data['rig_name']
        stop = data['config'].streams.tasklogic_input.data['task_parameters']['environment']['blocks'][0]['environment_statistics']['patches'][0]['reward_specification']['operant_logic']['stop_duration']['distribution_parameters']['value']
        all_epochs['stop'] = stop
        session_n +=1
        # if session_n > 15:
        #     continue
        stage = data['config'].streams.tasklogic_input.data['stage_name']
        if stage != 'mcm_final_stage' and stage != 'graduation' and stage != 'data_collection':
            continue
        all_epochs['stage'] = stage
        all_epochs['time_session'] = (all_epochs.index - all_epochs.index[0])/60
        list_of_epochs.append(all_epochs)
        
        vel_df = plotting.trial_collection(all_epochs.loc[(all_epochs.label == 'OdorSite')].reset_index(), stream_data.encoder_data, aligned='start_time', cropped_to_length='epoch')
        
        # Assign all new columns at once
        vel_df = vel_df.assign(
            mouse=mouse,
            session_date=session_path.name.split('_')[1][:10],
            session_n=session_n,
            rig=data['config'].streams.rig_input.data['rig_name']
        )
        agg_vel.append(vel_df)

vel_cum = pd.concat(agg_vel, ignore_index=True)
df = pd.concat(list_of_epochs, ignore_index=True)

In [ ]:
per_session =  df.groupby(['mouse', 'session_n', 'stage', 'rig', 'patch_number']).agg(total_rewards=('is_reward', 'sum'),
                                                                      total_choices=('site_number', 'max'),
                                                                      reward_amount = ('reward_amount', 'mean')).reset_index()


per_mouse = per_session.groupby(['mouse', 'session_n','rig','reward_amount']).agg(total_rewards=('total_rewards', 'mean'),
                                                        total_choices=('total_choices', 'mean')).reset_index()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metrics = ['total_choices', 'total_rewards']
titles = ['Total choices', 'Total rewards']


for ax, metric, title in zip(axes, metrics, titles):
    sns.boxplot(data=per_mouse, x='rig', y=metric, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Rig')
    ax.set_ylabel('')
    # ax.get_legend().remove()
    sns.despine(ax=ax)

axes[-1].legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

metrics = ['total_choices', 'total_rewards']
titles = ['Total choices', 'Total rewards']

for ax, metric, title in zip(axes, metrics, titles):
    sns.boxplot(data=per_mouse, x='mouse', y=metric, hue='rig', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Mouse')
    ax.set_ylabel('')
    ax.get_legend().remove()
    sns.despine(ax=ax)

axes[-1].legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.tight_layout()
plt.show()

In [ ]:
per_session =  df.groupby(['mouse', 'session_n', 'stage', 'rig']).agg(total_rewards=('is_reward', 'sum'),
                                                                      total_choices=('is_choice', 'sum'),
                                                                      reward_amount = ('reward_amount', 'mean')).reset_index()

per_session['total_water'] = per_session['total_rewards'] * per_session['reward_amount']
per_session['ratio'] = per_session['total_rewards'] / per_session['total_choices']

per_mouse = per_session.groupby(['mouse', 'session_n','reward_amount']).agg(total_rewards=('total_rewards', 'mean'),
                                                        total_choices=('total_choices', 'mean'),
                                                        total_water=('total_water', 'mean'),
                                                        ratio=('ratio', 'mean')).reset_index()

grand_mean = per_mouse.groupby(['session_n', 'reward_amount']).agg(total_rewards=('total_rewards', 'mean'),
                                                        total_choices=('total_choices', 'mean'),
                                                        total_water=('total_water', 'mean'),
                                                        ratio=('ratio', 'mean')).reset_index()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['total_choices', 'total_rewards', 'ratio']
titles = ['Total choices', 'Total rewards', 'Ratio']


for ax, metric, title in zip(axes, metrics, titles):
    sns.boxplot(data=per_session, x='rig', y=metric, hue='stage', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Rig')
    ax.set_ylabel('')
    ax.get_legend().remove()
    sns.despine(ax=ax)

axes[-1].legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['total_choices', 'total_rewards', 'ratio']
titles = ['Total choices', 'Total rewards', 'Ratio']

for ax, metric, title in zip(axes, metrics, titles):
    sns.boxplot(data=per_session, x='mouse', y=metric, hue='rig', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Mouse')
    ax.set_ylabel('')
    ax.get_legend().remove()
    sns.despine(ax=ax)

axes[-1].legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.tight_layout()
plt.show()

In [ ]:
def add_training_day(df, ephys_rig='ND_Ephys.5'):
    df = df.copy()
    df['session_date'] = pd.to_datetime(df['session_date'])

    session_order = (
        df.groupby('mouse')['session_date']
        .apply(lambda x: sorted(x.unique()))
        .reset_index()
        .explode('session_date')
    )
    session_order['session_date'] = pd.to_datetime(session_order['session_date'])
    session_order['training_day'] = session_order.groupby('mouse').cumcount()

    move_training_day = (
        session_order.merge(
            df[df['rig'] == ephys_rig]
            .groupby('mouse')['session_date'].min().reset_index(),
            on=['mouse', 'session_date']
        )
        .set_index('mouse')['training_day']
    )

    session_order['session_relative_to_ephys'] = (
        session_order['training_day'] - session_order['mouse'].map(move_training_day)
    )

    return df.merge(
        session_order[['mouse', 'session_date', 'session_relative_to_ephys']],
        on=['mouse', 'session_date'],
        how='left'
    )

df = add_training_day(df)
vel_cum = add_training_day(vel_cum)

In [ ]:
per_session =  df.loc[df.is_choice == 1].groupby(['mouse', 'session_relative_to_ephys', 'stage', 'rig']).agg(total_rewards=('is_reward', 'sum'),
                                                                      total_choices=('is_choice', 'sum'),
                                                                      reward_amount = ('reward_amount', 'mean')).reset_index()


fig, ax = plt.subplots(2,1, figsize=(10, 6), sharex=True)

sns.scatterplot(data=per_session, x='session_relative_to_ephys', y='total_choices', hue='mouse', ax=ax[0])
sns.scatterplot(data=per_session, x='session_relative_to_ephys', y='total_rewards', hue='mouse', ax=ax[1])
ax[1].set_xlabel('Session relative to ephys move')
ax[0].legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
ax[1].legend().remove()
sns.despine()
ax[0].vlines(0, ymin=0, ymax=per_session['total_choices'].max(), color='red', linestyle='--')
ax[1].vlines(0, ymin=0, ymax=per_session['total_rewards'].max(), color='red', linestyle='--')

ax[0].set_xlim(-11, 10)
ax[1].set_xlim(-11, 10)

In [ ]:
per_session =  df.loc[df.is_choice == 1].groupby(['mouse', 'session_relative_to_ephys', 'stage', 'rig', 'patch_number']).agg(total_rewards=('is_reward', 'sum'),
                                                                      total_choices=('site_number', 'max'),
                                                                      reward_amount = ('reward_amount', 'mean')).reset_index()


per_mouse = per_session.groupby(['mouse', 'session_relative_to_ephys','rig','reward_amount']).agg(total_rewards=('total_rewards', 'mean'),
                                                        total_choices=('total_choices', 'mean')).reset_index()

fig, ax = plt.subplots(2,1, figsize=(10, 6), sharex=True)

sns.scatterplot(data=per_mouse, x='session_relative_to_ephys', y='total_choices', hue='mouse', ax=ax[0])
sns.scatterplot(data=per_mouse, x='session_relative_to_ephys', y='total_rewards', hue='mouse', ax=ax[1])
ax[1].set_xlabel('Session relative to ephys move')
ax[0].legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
sns.despine()
ax[0].vlines(0, ymin=0, ymax=per_mouse['total_choices'].max(), color='red', linestyle='--')
ax[1].vlines(0, ymin=0, ymax=per_mouse['total_rewards'].max(), color='red', linestyle='--')

ax[0].set_xlim(-11, 10)
ax[1].set_xlim(-11, 10)

In [ ]:
per_session = df.loc[(df.is_choice == 1) & (df.stop == 1)].groupby(['session_relative_to_ephys', 'mouse', 'rig', 'patch_label', 'is_reward']).agg({'duration_epoch': 'median'}).reset_index()

fig, axes = plt.subplots(1, len(per_session['mouse'].unique()), figsize=(len(per_session['mouse'].unique()) * 4, 5), sharey=True)

for ax, mouse in zip(axes, per_session['mouse'].unique()):
    data = per_session[per_session['mouse'] == mouse]

    sns.boxplot(
        data=data.loc[(data.is_reward == 1) & (data.patch_label == 'odor_60')], x='rig', y='duration_epoch', 
        width=0.5, ax=ax
    )
    ax.set_title(f"Mouse {mouse}", fontsize=12)
    ax.set_xlabel('Label')
    # ax.get_legend().remove()    
    ax.set_ylabel('Mean duration' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=20)

sns.despine()
plt.tight_layout()

In [ ]:
per_session = df.groupby(['session_relative_to_ephys', 'mouse', 'rig', 'label']).agg({'duration_epoch': 'median'}).reset_index()

fig, axes = plt.subplots(1, len(per_session['mouse'].unique()), figsize=(len(per_session['mouse'].unique()) * 5, 5), sharey=True)

for ax, mouse in zip(axes, per_session['mouse'].unique()):
    data = per_session[per_session['mouse'] == mouse]

    sns.boxplot(
        data=data, x='label', y='duration_epoch', hue='rig', 
        width=0.5, ax=ax
    )
    ax.set_title(f"Mouse {mouse}", fontsize=12)
    ax.set_xlabel('Label')
    ax.get_legend().remove()    
    ax.set_ylabel('Mean duration' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=20)

sns.despine()
plt.tight_layout()

In [ ]:
per_session = vel_cum.groupby(['session_n', 'mouse', 'rig']).agg({'speed': 'mean'}).reset_index()
n_mice = per_session['mouse'].nunique()
fig, axes = plt.subplots(1, n_mice, figsize=(n_mice * 4, 5), sharey=True)

for ax, mouse in zip(axes, per_session['mouse'].unique()):
    data = per_session[per_session['mouse'] == mouse]

    sns.boxplot(
        data=data, x='rig', y='speed',
        boxprops=dict(alpha=0), whiskerprops=dict(alpha=0),
        capprops=dict(alpha=0), flierprops=dict(alpha=0),
        medianprops=dict(linewidth=2.5, color='black'),
        width=0.5, ax=ax
    )
    sns.swarmplot(
        data=data, x='rig', y='speed',
        palette='tab10', size=5, ax=ax
    )

    ax.set_title(f"Mouse {mouse}", fontsize=12)
    ax.set_xlabel('Rig')
    ax.set_ylabel('Mean speed' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=20)

sns.despine()
plt.tight_layout()

In [ ]:
n_mice = vel_cum['mouse'].nunique()
fig, axes = plt.subplots(1, n_mice, figsize=(n_mice * 6, 5), sharey=True)
for ax, mouse in zip(axes, vel_cum['mouse'].unique()):
    sns.lineplot(data=vel_cum.loc[(vel_cum['mouse']==mouse) & (vel_cum['is_choice']==1)& (vel_cum['is_reward']==0)& (vel_cum['session_relative_to_ephys']>=-2)& (vel_cum['session_relative_to_ephys']<=3)], x='times', y='speed', hue='session_relative_to_ephys', style='rig',errorbar=None, ax=ax)
    ax.set_xlim(0, 5)
    sns.despine(ax=ax)
    ax.set_title(f'Mouse {mouse}')
    ax.get_legend().remove()
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

In [ ]:
n_mice = vel_cum['mouse'].nunique()
fig, axes = plt.subplots(1, n_mice, figsize=(n_mice * 6, 5), sharey=True)
for ax, mouse in zip(axes, vel_cum['mouse'].unique()):
    sns.lineplot(data=vel_cum.loc[(vel_cum['mouse']==mouse) & (vel_cum['is_choice']==1)& (vel_cum['is_reward']==0)], x='times', y='speed', hue='rig', style='patch_label', errorbar=None, ax=ax)
    ax.set_xlim(0, 5)
    sns.despine(ax=ax)
    ax.set_title(f'Mouse {mouse}')
    ax.get_legend().remove()
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

In [ ]:
n_mice = vel_cum['mouse'].nunique()
fig, axes = plt.subplots(1, n_mice, figsize=(n_mice * 6, 5), sharey=True)
for ax, mouse in zip(axes, vel_cum['mouse'].unique()):
    sns.lineplot(data=vel_cum.loc[(vel_cum['mouse']==mouse) & (vel_cum['is_choice']==1)], x='times', y='speed', hue='is_reward', style='stop',errorbar=None, ax=ax)
    ax.set_xlim(0, 5)
    sns.despine(ax=ax)
    ax.set_title(f'Mouse {mouse}')

In [ ]:
n_mice = vel_cum['mouse'].nunique()
fig, axes = plt.subplots(1, n_mice, figsize=(n_mice * 6, 5), sharey=True)
for ax, mouse in zip(axes, vel_cum['mouse'].unique()):
    sns.lineplot(data=vel_cum.loc[(vel_cum['mouse']==mouse) & (vel_cum['is_reward']==0)& (vel_cum['is_choice']==1)], x='times', y='speed', hue='rig', errorbar=None, ax=ax)
    ax.set_xlim(0, 5)
    sns.despine(ax=ax)
    ax.set_title(f'Mouse {mouse}')

In [ ]:
sns.lineplot(data=vel_cum.loc[(vel_cum['is_reward']==1)], x='times', y='speed', hue='mouse', style='reward_amount', errorbar=None)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

In [ ]:
n_mice = vel_cum['mouse'].nunique()
fig, axes = plt.subplots(1, n_mice, figsize=(n_mice * 6, 5), sharey=True)
for ax, mouse in zip(axes, vel_cum['mouse'].unique()):
    sns.lineplot(data=vel_cum.loc[(vel_cum['mouse']==mouse)], x='times', y='speed', hue='rig', errorbar='sd', ax=ax)
    # ax.set_xlim(0, 5)
    sns.despine(ax=ax)
    ax.set_title(f'Mouse {mouse}')

## Analysis

In [ ]:
# Map stages to ordered ranks
stage_order = ['one_odor_no_depletion','one_odor_w_depletion_day_0','one_odor_w_depletion_day_1','all_odors_rewarded','Graduation', 'mcm_final_stage']  # explicitly ordered
stage_to_rank = {s: i for i, s in enumerate(stage_order)}
df['stage_rank'] = df['stage'].map(stage_to_rank)

# One row per mouse/session
session_stage = (
    df.groupby(['mouse', 'session_n'])['stage']
    .first()
    .reset_index()
)

# session_stage = session_stage.loc[session_stage.session_n > 1]
# session_stage['session_n'] -= 1
session_stage['stage_rank'] = session_stage['stage'].map(stage_to_rank)

# Add graduation stage for each mouse (one session after their last)
last_sessions = session_stage.groupby('mouse')['session_n'].max().reset_index()
# last_sessions['session_n'] = last_sessions['session_n'] + 1
last_sessions['stage'] = 'Graduation'
session_stage_plot = pd.concat([session_stage, last_sessions], ignore_index=True)

# Sequential color palette for progression
n_stages = len(stage_order)

palette_div = sns.diverging_palette(220, 20, n=n_stages)
# stage_palette = dict(zip(stage_order, palette_div))

# Explicit stage order
stage_order = ['one_odor_no_depletion','one_odor_w_depletion_day_0','one_odor_w_depletion_day_1','all_odors_rewarded','Graduation', 'mcm_final_stage']

# Custom colors
stage_palette = {
    'one_odor_no_depletion': "#E0E0E0",          # blue
    'one_odor_w_depletion_day_0': "#AEACAC",          # red
    'one_odor_w_depletion_day_1': "#828282",  
    'all_odors_rewarded': "#5E5E5E",  # green
    'Graduation': "k",  # black
    'graduation': "k",  # black
    'mcm_final_stage': "k"
}
# Map mice to y-axis
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
mice = session_stage_plot['mouse'].unique()
mouse_to_y = {m: i for i, m in enumerate(mice)}

# Remap to consecutive session index per mouse (no gaps)
session_stage_plot = session_stage_plot.sort_values(['mouse', 'session_n'])
session_stage_plot['session_rel'] = session_stage_plot.groupby('mouse').cumcount() + 1

# Plot Gantt-like bars
for _, row in session_stage_plot.iterrows():
    ax.barh(
        y=mouse_to_y[row['mouse']],
        width=1,
        left=row['session_rel'] - 1,
        color=stage_palette[row['stage']],
        edgecolor='white',
        linewidth=0.5
    )

# Labels
ax.set_yticks(range(len(mice)))
ax.set_yticklabels(mice)
ax.set_xlabel('Session')
ax.set_ylabel('Mouse')
ax.set_title('Stage progression (Gantt)')

# Legend
handles = [mpatches.Patch(color=stage_palette[s], label=s) for s in stage_order]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)

ax.set_xlim(0, session_stage_plot['session_rel'].max() + 1)
sns.despine(ax=ax)
plt.xlim(-1, 15)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'stage_progression_gantt_all_mice.pdf'), dpi=300, bbox_inches='tight')

In [ ]:
import numpy as np
df['stage_num'] = df['stage'].map(stage_to_rank)
df.fillna({'stage_num': stage_to_rank['Graduation']}, inplace=True)

fig, ax = plt.subplots(figsize=(4,4))

mice = df['mouse'].unique()
offsets = np.linspace(-0.15, 0.15, len(mice))
mouse_offset = dict(zip(mice, offsets))

# grayscale palette: dark to light
grays = np.linspace(0.2, 0.8, len(mice))

# individual mice
for (mouse, g), gray in zip(df.sort_values('session_n').groupby('mouse'), grays):
    ax.step(
        g['session_n'],
        g['stage_num'],
        where='post',
        color=str(gray),
        alpha=0.2
    )

# mean progression
mean_stage = df.groupby('session_n')['stage_num'].mean().sort_index()

ax.plot(
    mean_stage.index,
    mean_stage.values,
    linewidth=3,
    color='k',
)

# vertical line at average session when each mouse first reaches Graduation or MCM_final_stage
graduation_rank = stage_to_rank['Graduation']
mcm_final_rank = stage_to_rank['mcm_final_stage']
threshold = min(graduation_rank, mcm_final_rank)

first_graduation = (
    df[df['stage_num'] >= threshold]
    .groupby('mouse')['session_n']
    .min()
)

avg_graduation_session = first_graduation.mean()-1

# ax.axvline(avg_graduation_session, color='red', linestyle='--', linewidth=1.5, 
#            label=f'Avg. graduation (session {avg_graduation_session:.1f})')
# ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7).remove()

ax.set_yticks(range(len(stage_order)))
ax.set_yticklabels(stage_order)
ax.set_xlim(0, 18)

ax.set_xlabel("Session")
ax.set_ylabel("Stage")
sns.despine()

In [ ]:
summary_df = df.groupby(['mouse', 'session_n', 'rig']).agg({"length": "sum",
                                                        "is_reward": "sum",
                                                        "site_number": "mean",
                                                        "is_choice": "sum"
                                                        }).reset_index()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.scatterplot(x='session_n', y='length', data=summary_df, hue='rig',  ax=ax)
ax.set_xlabel("Session")
ax.set_ylabel("Total length (cm)")
plt.legend(title='Rig', bbox_to_anchor=(1.25, 1), loc='upper left')

ax2 = ax.twinx()
sns.lineplot(x='session_n', y='is_choice', data=summary_df, marker='o', color='orange', ax=ax2)
sns.lineplot(x='session_n', y='is_reward', data=summary_df, marker='o', color='darkgreen', ax=ax2)
plt.xlim(0, 15.5)
sns.despine()


In [ ]:
plot = agg_df.groupby(['session_date', 'times', 'is_choice', 'patch_label', 'site_number', 'is_reward']).agg({'speed': 'mean'}).reset_index()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.lineplot(x='times', y='speed', hue='session_date', data=agg_df, palette='tab20')
sns.despine()
plt.legend(title='Rig', bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
from matplotlib.colors import to_rgba
import colorsys

variables = ['length', 'is_reward', 'site_number', 'is_choice']
labels = ['Total length (cm)', 'Rewards', 'Average stops (s)', 'Stops']

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=False)

# Assign a distinct hue per mouse
mice = summary_df['mouse'].unique()
base_colors = sns.color_palette('tab10', n_colors=len(mice))
mouse_base = dict(zip(mice, base_colors))

# Consistent rig order
rig_order = sorted(summary_df['rig'].unique())
rig_to_x = {r: i for i, r in enumerate(rig_order)}

# For each mouse, grade lightness by session_n (early = light, late = dark)
session_min = summary_df['session_n'].min()
session_max = summary_df['session_n'].max()

def graded_color(base_rgb, session_n, smin, smax):
    """Lighter for early sessions, full saturation for late."""
    if smax == smin:
        return base_rgb
    t = (session_n - smin) / (smax - smin)
    r, g, b = base_rgb[:3]
    factor = 0.3 + 0.7 * t
    return (1 - factor + factor * r, 1 - factor + factor * g, 1 - factor + factor * b)

# Build per-row color
summary_df['_color'] = summary_df.apply(
    lambda row: graded_color(mouse_base[row['mouse']], row['session_n'], session_min, session_max),
    axis=1
)

for ax, var, label in zip(axes.flat, variables, labels):
    # Boxplot with no outline, using same rig order
    sns.boxplot(
        x='rig', y=var, data=summary_df, color='lightgray',
        order=rig_order,
        ax=ax, fliersize=0,
        boxprops=dict(edgecolor='none'),
        whiskerprops=dict(color='gray', linewidth=0.5),
        capprops=dict(color='gray', linewidth=0.5),
        medianprops=dict(color='dimgray', linewidth=1.5),
    )
    # Scatter points using the same rig order
    for _, row in summary_df.iterrows():
        ax.scatter(
            rig_to_x[row['rig']] + np.random.uniform(-0.15, 0.15),
            row[var],
            color=row['_color'],
            edgecolor='white', linewidth=0.3,
            s=40, alpha=0.8, zorder=3
        )
    ax.set_ylabel(label)
    ax.set_xlabel('Rig')

# Legend: one entry per mouse with base color
handles = [mpatches.Patch(color=mouse_base[m], label=m) for m in mice]
fig.legend(handles=handles, bbox_to_anchor=(1.01, 0.95), loc='upper left', fontsize=7, title='Mouse\n(lighter = earlier session)')

sns.despine()
plt.tight_layout()
